# EPOWER Energy Intelligence Demo Setup

**A Hands-On Lab for Snowflake Intelligence**

---

## What is Snowflake Intelligence?

**Snowflake Intelligence** is a ready-to-use agentic AI application with an intuitive, conversational interface that helps business users discover and act on deep insights. It lets users interact with their structured and unstructured enterprise data using natural language.

Snowflake Intelligence uses AI-powered **data agents** to:
- **Understand** questions in natural language
- **Perform** analysis across structured and unstructured data
- **Generate** trusted insights with full traceability
- **Take action** on behalf of users

It bridges the gap between valuable enterprise data and the people who need it, empowering users to move beyond stale dashboards and rigid reports - finding answers independently while respecting Snowflake's robust security and governance policies.

### How It Works

```
User Question ──► Cortex Agent API ──► Orchestration (LLM) ──► Tool Selection
                                                                     │
                      ┌──────────────────┬──────────────────┬────────┘
                      ▼                  ▼                  ▼
              ┌──────────────┐   ┌──────────────┐   ┌──────────────┐
              │CORTEX ANALYST│   │CORTEX SEARCH │   │ CUSTOM TOOLS │
              │ (Text-to-SQL)│   │    (RAG)     │   │  (Functions) │
              │              │   │              │   │              │
              │ Semantic     │   │ Documents &  │   │ Web Scraper  │
              │ Views        │   │ Unstructured │   │ Procedures   │
              └──────────────┘   └──────────────┘   └──────────────┘
                      │                  │                  │
                      └──────────────────┴──────────────────┘
                                         │
                                         ▼
                            Reflection & Response Generation
```

---

## What You'll Build

This notebook guides you through building a complete **Snowflake Intelligence** demo for **EPOWER**, a fictional German Energy Retail (B2C) use case. You will create:

| Component | Purpose |
|-----------|---------|
| **Governed Data Foundation** | Structured tables managed and secured by Snowflake |
| **Semantic Views** | Business context layer enabling natural language queries |
| **Cortex Search Services** | RAG over documents and service tickets |
| **Intelligence Agent** | Autonomous AI agent orchestrating all capabilities |

### How to Use This Notebook

You can either:
- **Run all cells** sequentially for a quick setup (~10 minutes)
- **Run cell-by-cell** to learn about each concept as you go

### Prerequisites

- ACCOUNTADMIN role access (or equivalent privileges)
- A Snowflake account with Cortex features enabled

## Session Setup

Before running any cells, we need to establish a connection to Snowflake. In a Snowflake Workspace, use `get_active_session()` to get the current session context.

In [3]:
# Import python packages
import streamlit as st
import pandas as pd

# Snowpark
from snowflake.snowpark.context import get_active_session
import snowflake.snowpark.functions as F

# Cortex Functions
import snowflake.cortex as cortex

# Get the active session
session = get_active_session()

ModuleNotFoundError: No module named 'snowflake.cortex'

## Table of Contents

1. [Introduction to EPOWER Demo](#1-introduction)
2. [Role & Warehouse Setup](#2-role-warehouse-setup)
3. [Database & Schema Creation](#3-database-schema)
4. [Data Model - Dimension Tables](#4-dimension-tables)
5. [Data Model - Fact Tables](#5-fact-tables)
6. [Data Loading from Workspace Files](#6-data-loading)
7. [Semantic Views for Cortex Analyst](#7-semantic-views)
8. [Unstructured Data Processing](#8-unstructured-data)
9. [Cortex Search Services](#9-cortex-search)
10. [Snowflake Intelligence Agent](#10-agent)
11. [Verification & Next Steps](#11-verification)

---

## 1. Introduction to EPOWER Demo <a id="1-introduction"></a>

### The Business Scenario

**EPOWER** is a fictional German energy provider offering:

| Product Category | Description |
|-----------------|-------------|
| **Power & Gas** | Traditional electricity and gas tariffs |
| **Future Energy Home** | Solar panels, heat pumps, battery storage |
| **Smart Home** | Smart meters, energy management systems |
| **E-Mobility** | Wallbox charging stations, EV tariffs |

### What We're Building

```
┌─────────────────────────────────────────────────────────────────────────┐
│                     SNOWFLAKE INTELLIGENCE AGENT                        │
│                    (Natural Language Interface)                         │
└─────────────────────────────────────────────────────────────────────────┘
                                    │
                 ┌──────────────────┼──────────────────┐
                 ▼                  ▼                  ▼
        ┌────────────────┐  ┌────────────────┐  ┌────────────────┐
        │ CORTEX ANALYST │  │ CORTEX SEARCH  │  │  WEB SCRAPER   │
        │  (Text-to-SQL) │  │     (RAG)      │  │  (Live Data)   │
        └────────────────┘  └────────────────┘  └────────────────┘
```

### Key Snowflake Features Demonstrated

| Feature | Purpose |
|---------|----------|
| **Git Integration** | Version-controlled data and code |
| **Semantic Views** | Business-friendly data modeling for AI |
| **Cortex Analyst** | Natural language to SQL conversion |
| **Cortex Search** | Vector search over documents |
| **Cortex Parse** | PDF/document parsing |
| **Intelligence Agent** | Multi-tool AI orchestration |

---

## 2. Role & Warehouse Setup <a id="2-role-warehouse-setup"></a>

### Concept: Role-Based Access Control (RBAC)

Snowflake uses **roles** to manage permissions. We'll create a dedicated role for this demo that has all necessary privileges while following the principle of least privilege.

**Best Practice**: Always use a dedicated role for demos/projects rather than ACCOUNTADMIN for day-to-day work.

In [2]:
-- Switch to accountadmin to create resources
USE ROLE accountadmin;

SyntaxError: invalid syntax (2377990862.py, line 1)

In [ ]:
-- Create Snowflake Intelligence object (account-level resource for agents)
CREATE SNOWFLAKE INTELLIGENCE IF NOT EXISTS snowflake_intelligence_object_default;

In [ ]:
-- Create a dedicated role for this demo
CREATE OR REPLACE ROLE Energy_Intelligence_Demo;

-- Grant the role to current user
SET current_user_name = CURRENT_USER();
GRANT ROLE Energy_Intelligence_Demo TO USER IDENTIFIER($current_user_name);

-- Grant necessary account-level privileges
GRANT CREATE DATABASE ON ACCOUNT TO ROLE Energy_Intelligence_Demo;

### Concept: Virtual Warehouses

A **Virtual Warehouse** provides the compute resources to execute queries. Key features:

- **AUTO_SUSPEND**: Automatically pauses after idle time (saves credits)
- **AUTO_RESUME**: Automatically starts when queries arrive
- **Elastic Scaling**: Can resize without downtime

We use **XSMALL** for this demo - sufficient for our data volumes and cost-effective.

In [ ]:
-- Create a dedicated warehouse for the demo
CREATE OR REPLACE WAREHOUSE Energy_Intelligence_demo_wh 
    WITH WAREHOUSE_SIZE = 'XSMALL'
    AUTO_SUSPEND = 300
    AUTO_RESUME = TRUE;

GRANT USAGE ON WAREHOUSE ENERGY_INTELLIGENCE_DEMO_WH TO ROLE Energy_Intelligence_Demo;

In [ ]:
-- Set default role and warehouse for convenience
ALTER USER IDENTIFIER($current_user_name) SET DEFAULT_ROLE = Energy_Intelligence_Demo;
ALTER USER IDENTIFIER($current_user_name) SET DEFAULT_WAREHOUSE = Energy_Intelligence_demo_wh;

-- Switch to demo role
USE ROLE Energy_Intelligence_Demo;

---

## 3. Database & Schema Creation <a id="3-database-schema"></a>

### Concept: Database Organization

Snowflake uses a three-tier namespace: `DATABASE.SCHEMA.OBJECT`

```
ENERGY_AI_DEMO (Database)
    └── ENERGY_SCHEMA (Schema)
            ├── Tables
            ├── Semantic Views
            ├── Cortex Search Services
            └── Functions/Procedures
```

In [ ]:
-- Create database and schema
CREATE OR REPLACE DATABASE ENERGY_AI_DEMO;
USE DATABASE ENERGY_AI_DEMO;

CREATE SCHEMA IF NOT EXISTS ENERGY_SCHEMA;
USE SCHEMA ENERGY_SCHEMA;

### File Format Definition

A **File Format** defines how Snowflake should parse incoming data files. This is essential for CSV ingestion.

In [ ]:
-- Create file format for CSV files
CREATE OR REPLACE FILE FORMAT CSV_FORMAT
    TYPE = 'CSV'
    FIELD_DELIMITER = ','
    RECORD_DELIMITER = '\n'
    SKIP_HEADER = 1
    FIELD_OPTIONALLY_ENCLOSED_BY = '"'
    TRIM_SPACE = TRUE
    ERROR_ON_COLUMN_COUNT_MISMATCH = FALSE
    ESCAPE = 'NONE'
    ESCAPE_UNENCLOSED_FIELD = '\\'
    DATE_FORMAT = 'YYYY-MM-DD'
    TIMESTAMP_FORMAT = 'YYYY-MM-DD HH24:MI:SS'
    NULL_IF = ('NULL', 'null', '', 'N/A', 'n/a');

### Internal Stage for Unstructured Documents

**Why do we need a stage here?**

In a Git-connected Snowflake Workspace, files from the repository are accessible to **Python code** via local paths (e.g., `pd.read_csv("data/file.csv")`). However, **SQL functions like `AI_PARSE_DOCUMENT`** can only read files from Snowflake stages - they cannot access workspace local files directly.

Therefore, for **Cortex Parse** (PDF/document processing), we must:
1. Create an internal stage with directory tables enabled
2. Upload the unstructured documents from the workspace to the stage
3. Then use `AI_PARSE_DOCUMENT` to extract content from the staged files

This is the only step where staging is required - all CSV data loading uses Python's direct file access.

In [ ]:
-- Create internal stage for unstructured documents (needed for Cortex Parse)
CREATE OR REPLACE STAGE ENERGY_STAGE
    FILE_FORMAT = CSV_FORMAT
    COMMENT = 'Internal stage for Energy demo unstructured documents'
    DIRECTORY = (ENABLE = TRUE)
    ENCRYPTION = (TYPE = 'SNOWFLAKE_SSE');

In [ ]:
# Upload unstructured documents from workspace to internal stage
import glob

unstructured_path = "unstructured_docs"
files = glob.glob(f"{unstructured_path}/*")

for file_path in files:
    session.file.put(file_path, "@ENERGY_STAGE/unstructured_docs/", auto_compress=False, overwrite=True)
    print(f"✓ Uploaded {file_path}")

session.sql("ALTER STAGE ENERGY_STAGE REFRESH").collect()
print("✓ Stage refreshed")

---

## 5. Data Model - Dimension Tables <a id="5-dimension-tables"></a>

### Concept: Star Schema Design

We use a **Star Schema** - a classic data warehouse design pattern:

- **Dimension Tables**: Descriptive attributes (WHO, WHAT, WHERE, WHEN)
- **Fact Tables**: Measurable events/transactions (HOW MUCH, HOW MANY)

**Benefits:**
- Simple, intuitive queries
- Optimized for analytical workloads
- Perfect for Semantic Views and Cortex Analyst

In [ ]:
-- Product Category Dimension
CREATE OR REPLACE TABLE product_category_dim (
    category_key INT PRIMARY KEY,
    category_name VARCHAR(100) NOT NULL,
    vertical VARCHAR(50) NOT NULL
) COMMENT = 'Energy product categories: Electricity, Gas, Solar, Heat Pumps, Smart Home, E-Mobility';

In [ ]:
-- Product Dimension (Energy products/tariffs)
CREATE OR REPLACE TABLE product_dim (
    product_key INT PRIMARY KEY,
    product_name VARCHAR(200) NOT NULL,
    category_key INT NOT NULL,
    category_name VARCHAR(100),
    vertical VARCHAR(50)
) COMMENT = 'Energy products: EPOWER Strom, Gas, Solar, Wärmepumpe offerings';

In [ ]:
-- Customer Dimension (German residential and business customers)
CREATE OR REPLACE TABLE customer_dim (
    customer_key INT PRIMARY KEY,
    customer_name VARCHAR(200) NOT NULL,
    customer_type VARCHAR(50),
    housing_type VARCHAR(100),
    vertical VARCHAR(50),
    address VARCHAR(200),
    city VARCHAR(100),
    state VARCHAR(50),
    zip VARCHAR(20),
    region_key INT
) COMMENT = 'German energy customers with housing type for consumption analysis';

In [ ]:
-- Vendor Dimension (Installation and service partners)
CREATE OR REPLACE TABLE vendor_dim (
    vendor_key INT PRIMARY KEY,
    vendor_name VARCHAR(200) NOT NULL,
    vendor_type VARCHAR(100),
    vertical VARCHAR(50),
    address VARCHAR(200),
    city VARCHAR(100),
    state VARCHAR(50),
    zip VARCHAR(20)
) COMMENT = 'Installation partners, service providers, and suppliers';

In [ ]:
-- Additional Dimension Tables
CREATE OR REPLACE TABLE account_dim (
    account_key INT PRIMARY KEY,
    account_name VARCHAR(100) NOT NULL,
    account_type VARCHAR(50)
);

CREATE OR REPLACE TABLE department_dim (
    department_key INT PRIMARY KEY,
    department_name VARCHAR(100) NOT NULL
) COMMENT = 'EPOWER organizational departments';

CREATE OR REPLACE TABLE region_dim (
    region_key INT PRIMARY KEY,
    region_name VARCHAR(100) NOT NULL
) COMMENT = 'German geographic regions: North, South, West, East';

CREATE OR REPLACE TABLE sales_rep_dim (
    sales_rep_key INT PRIMARY KEY,
    rep_name VARCHAR(200) NOT NULL,
    hire_date DATE
) COMMENT = 'Energy consultants and sales representatives';

CREATE OR REPLACE TABLE campaign_dim (
    campaign_key INT PRIMARY KEY,
    campaign_name VARCHAR(300) NOT NULL,
    objective VARCHAR(100)
) COMMENT = 'Marketing campaigns for energy products';

CREATE OR REPLACE TABLE channel_dim (
    channel_key INT PRIMARY KEY,
    channel_name VARCHAR(100) NOT NULL
);

CREATE OR REPLACE TABLE employee_dim (
    employee_key INT PRIMARY KEY,
    employee_name VARCHAR(200) NOT NULL,
    gender VARCHAR(1),
    hire_date DATE
);

CREATE OR REPLACE TABLE job_dim (
    job_key INT PRIMARY KEY,
    job_title VARCHAR(100) NOT NULL,
    job_level INT
);

CREATE OR REPLACE TABLE location_dim (
    location_key INT PRIMARY KEY,
    location_name VARCHAR(200) NOT NULL
);

---

## 6. Data Model - Fact Tables <a id="6-fact-tables"></a>

### Concept: Fact Tables

Fact tables contain:
- **Metrics/Measures**: Numeric values (amount, quantity, consumption)
- **Foreign Keys**: Links to dimension tables
- **Timestamps**: When the event occurred

Our EPOWER demo has **6 fact tables** covering:

| Fact Table | Business Domain |
|------------|----------------|
| `sales_fact` | Energy contracts and sales |
| `billing_history` | Consumption and invoicing |
| `service_logs` | Customer service tickets |
| `finance_transactions` | Financial operations |
| `marketing_campaign_fact` | Campaign performance |
| `hr_employee_fact` | Human resources |

In [ ]:
-- Sales Fact Table (Energy contracts)
CREATE OR REPLACE TABLE sales_fact (
    sale_id INT PRIMARY KEY,
    date DATE NOT NULL,
    customer_key INT NOT NULL,
    product_key INT NOT NULL,
    sales_rep_key INT NOT NULL,
    region_key INT NOT NULL,
    vendor_key INT NOT NULL,
    amount DECIMAL(12,2) NOT NULL,
    units INT NOT NULL
) COMMENT = 'Energy contracts and product sales. Amount in EUR, Units = kWh for tariffs or count for hardware';

In [ ]:
-- Billing History Table (Energy-specific)
CREATE OR REPLACE TABLE billing_history (
    billing_id INT PRIMARY KEY,
    customer_key INT NOT NULL,
    billing_date DATE NOT NULL,
    billing_type VARCHAR(50) NOT NULL,
    consumption_kwh INT NOT NULL,
    amount DECIMAL(10,2) NOT NULL,
    payment_status VARCHAR(50)
) COMMENT = 'Monthly energy consumption and billing. billing_type = Electricity or Gas';

In [ ]:
-- Service Logs Table (Customer service tickets)
CREATE OR REPLACE TABLE service_logs (
    log_id INT PRIMARY KEY,
    customer_key INT NOT NULL,
    log_date DATE NOT NULL,
    topic VARCHAR(100) NOT NULL,
    category VARCHAR(100),
    description VARCHAR(500),
    sentiment VARCHAR(50),
    channel VARCHAR(50),
    priority VARCHAR(50),
    resolution_date DATE,
    agent_key INT
) COMMENT = 'Customer service tickets. Sentiment = Positiv/Neutral/Negativ';

In [ ]:
-- Finance, Marketing, and HR Fact Tables
CREATE OR REPLACE TABLE finance_transactions (
    transaction_id INT PRIMARY KEY,
    date DATE NOT NULL,
    account_key INT NOT NULL,
    department_key INT NOT NULL,
    vendor_key INT NOT NULL,
    product_key INT NOT NULL,
    customer_key INT NOT NULL,
    amount DECIMAL(12,2) NOT NULL,
    approval_status VARCHAR(20) DEFAULT 'Pending',
    procurement_method VARCHAR(50),
    approver_id INT,
    approval_date DATE,
    purchase_order_number VARCHAR(50),
    contract_reference VARCHAR(100)
);

CREATE OR REPLACE TABLE marketing_campaign_fact (
    campaign_fact_id INT PRIMARY KEY,
    date DATE NOT NULL,
    campaign_key INT NOT NULL,
    product_key INT NOT NULL,
    channel_key INT NOT NULL,
    region_key INT NOT NULL,
    spend DECIMAL(10,2) NOT NULL,
    leads_generated INT NOT NULL,
    impressions INT NOT NULL
);

CREATE OR REPLACE TABLE hr_employee_fact (
    hr_fact_id INT PRIMARY KEY,
    date DATE NOT NULL,
    employee_key INT NOT NULL,
    department_key INT NOT NULL,
    job_key INT NOT NULL,
    location_key INT NOT NULL,
    salary DECIMAL(10,2) NOT NULL,
    attrition_flag INT NOT NULL
);

In [ ]:
-- Salesforce CRM Tables
CREATE OR REPLACE TABLE sf_accounts (
    account_id VARCHAR(20) PRIMARY KEY,
    account_name VARCHAR(200) NOT NULL,
    customer_key INT NOT NULL,
    industry VARCHAR(100),
    vertical VARCHAR(50),
    billing_street VARCHAR(200),
    billing_city VARCHAR(100),
    billing_state VARCHAR(50),
    billing_postal_code VARCHAR(20),
    account_type VARCHAR(50),
    annual_revenue DECIMAL(15,2),
    employees INT,
    created_date DATE
);

CREATE OR REPLACE TABLE sf_opportunities (
    opportunity_id VARCHAR(20) PRIMARY KEY,
    sale_id INT,
    account_id VARCHAR(20) NOT NULL,
    opportunity_name VARCHAR(200) NOT NULL,
    stage_name VARCHAR(100) NOT NULL,
    amount DECIMAL(15,2) NOT NULL,
    probability DECIMAL(5,2),
    close_date DATE,
    created_date DATE,
    lead_source VARCHAR(100),
    type VARCHAR(100),
    campaign_id INT
);

CREATE OR REPLACE TABLE sf_contacts (
    contact_id VARCHAR(20) PRIMARY KEY,
    opportunity_id VARCHAR(20) NOT NULL,
    account_id VARCHAR(20) NOT NULL,
    first_name VARCHAR(100),
    last_name VARCHAR(100),
    email VARCHAR(200),
    phone VARCHAR(50),
    title VARCHAR(100),
    department VARCHAR(100),
    lead_source VARCHAR(100),
    campaign_no INT,
    created_date DATE
);

---

## 7. Data Loading from Workspace Files <a id="7-data-loading"></a>

### Concept: Loading Data in Snowflake Workspaces

When running notebooks in a **Snowflake Workspace** connected to a Git repository, your files are directly accessible. We use **Python with Snowpark** to:

1. Read CSV files from the workspace directory
2. Create DataFrames
3. Write data to Snowflake tables

This approach leverages the native Workspace + Git integration without requiring separate API integrations or Git repository objects.

In [ ]:
# Load Dimension Tables from workspace files
import os

data_path = "demo_data"

dimension_tables = [
    "product_category_dim", "product_dim", "customer_dim", "vendor_dim",
    "account_dim", "department_dim", "region_dim", "sales_rep_dim",
    "campaign_dim", "channel_dim", "employee_dim", "job_dim", "location_dim"
]

for table_name in dimension_tables:
    file_path = f"{data_path}/{table_name}.csv"
    df = pd.read_csv(file_path)
    session.write_pandas(df, table_name.upper(), auto_create_table=False, overwrite=True)
    print(f"✓ Loaded {table_name}: {len(df)} rows")

In [ ]:
# Load Fact Tables
fact_tables = [
    "sales_fact", "billing_history", "service_logs",
    "finance_transactions", "marketing_campaign_fact", "hr_employee_fact"
]

for table_name in fact_tables:
    file_path = f"{data_path}/{table_name}.csv"
    df = pd.read_csv(file_path)
    session.write_pandas(df, table_name.upper(), auto_create_table=False, overwrite=True)
    print(f"✓ Loaded {table_name}: {len(df)} rows")

In [ ]:
# Load Salesforce CRM Tables
sf_tables = ["sf_accounts", "sf_opportunities", "sf_contacts"]

for table_name in sf_tables:
    file_path = f"{data_path}/{table_name}.csv"
    df = pd.read_csv(file_path)
    session.write_pandas(df, table_name.upper(), auto_create_table=False, overwrite=True)
    print(f"✓ Loaded {table_name}: {len(df)} rows")

In [ ]:
-- Verify data loading
SELECT 'DIMENSION TABLES' as category, '' as table_name, NULL as row_count
UNION ALL SELECT '', 'product_category_dim', COUNT(*) FROM product_category_dim
UNION ALL SELECT '', 'product_dim', COUNT(*) FROM product_dim
UNION ALL SELECT '', 'customer_dim', COUNT(*) FROM customer_dim
UNION ALL SELECT '', 'vendor_dim', COUNT(*) FROM vendor_dim
UNION ALL SELECT '', 'region_dim', COUNT(*) FROM region_dim
UNION ALL SELECT '', '', NULL
UNION ALL SELECT 'FACT TABLES', '', NULL
UNION ALL SELECT '', 'sales_fact', COUNT(*) FROM sales_fact
UNION ALL SELECT '', 'billing_history', COUNT(*) FROM billing_history
UNION ALL SELECT '', 'service_logs', COUNT(*) FROM service_logs
UNION ALL SELECT '', 'finance_transactions', COUNT(*) FROM finance_transactions
UNION ALL SELECT '', 'marketing_campaign_fact', COUNT(*) FROM marketing_campaign_fact
UNION ALL SELECT '', 'hr_employee_fact', COUNT(*) FROM hr_employee_fact;

---

## 8. Semantic Views for Cortex Analyst <a id="8-semantic-views"></a>

### Concept: Semantic Views

**Semantic Views** are a powerful Snowflake feature that bridges the gap between technical data models and business understanding. They enable:

1. **Natural Language Queries**: Ask questions in plain German or English
2. **Business Vocabulary**: Define synonyms (e.g., "Kunden" = "customers")
3. **Pre-defined Metrics**: Common calculations ready to use
4. **Relationship Mapping**: Automatic joins between tables

**How Cortex Analyst Uses Semantic Views:**

```
User Question: "What were total sales by region last quarter?"
       |
       v
+--------------------+
|   CORTEX ANALYST   |
| (LLM + Semantic    |
|     Context)       |
+--------------------+
          |
          v
   Generated SQL Query
```

We'll create **4 Semantic Views** for different business domains.

In [ ]:
-- ENERGY SALES SEMANTIC VIEW (Contracts and Products)
CREATE OR REPLACE SEMANTIC VIEW ENERGY_AI_DEMO.ENERGY_SCHEMA.ENERGY_SALES_SEMANTIC_VIEW
    tables (
        CUSTOMERS as ENERGY_AI_DEMO.ENERGY_SCHEMA.CUSTOMER_DIM primary key (CUSTOMER_KEY) 
            with synonyms=('Kunden','customers','Privatkunden','Gewerbekunden') 
            comment='German energy customers with housing type for consumption analysis',
        PRODUCTS as ENERGY_AI_DEMO.ENERGY_SCHEMA.PRODUCT_DIM primary key (PRODUCT_KEY) 
            with synonyms=('Produkte','Tarife','products','tariffs') 
            comment='Energy products: Strom, Gas, Solar, Waermepumpe, Smart Home, E-Mobility',
        PRODUCT_CATEGORIES as ENERGY_AI_DEMO.ENERGY_SCHEMA.PRODUCT_CATEGORY_DIM primary key (CATEGORY_KEY)
            with synonyms=('Kategorien','categories')
            comment='Product categories: Electricity, Gas, Solar, Heat Pumps, Smart Home, E-Mobility',
        REGIONS as ENERGY_AI_DEMO.ENERGY_SCHEMA.REGION_DIM primary key (REGION_KEY) 
            with synonyms=('Regionen','regions','Gebiete') 
            comment='German regions: North, South, West, East',
        CONTRACTS as ENERGY_AI_DEMO.ENERGY_SCHEMA.SALES_FACT primary key (SALE_ID) 
            with synonyms=('Vertraege','contracts','sales','Auftraege') 
            comment='Energy contracts and product sales',
        SALES_REPS as ENERGY_AI_DEMO.ENERGY_SCHEMA.SALES_REP_DIM primary key (SALES_REP_KEY) 
            with synonyms=('Berater','Energieberater','consultants') 
            comment='Energy consultants',
        VENDORS as ENERGY_AI_DEMO.ENERGY_SCHEMA.VENDOR_DIM primary key (VENDOR_KEY) 
            with synonyms=('Partner','Installateure','vendors','suppliers') 
            comment='Installation and service partners'
    )
    relationships (
        PRODUCT_TO_CATEGORY as PRODUCTS(CATEGORY_KEY) references PRODUCT_CATEGORIES(CATEGORY_KEY),
        CONTRACTS_TO_CUSTOMERS as CONTRACTS(CUSTOMER_KEY) references CUSTOMERS(CUSTOMER_KEY),
        CONTRACTS_TO_PRODUCTS as CONTRACTS(PRODUCT_KEY) references PRODUCTS(PRODUCT_KEY),
        CONTRACTS_TO_REGIONS as CONTRACTS(REGION_KEY) references REGIONS(REGION_KEY),
        CONTRACTS_TO_REPS as CONTRACTS(SALES_REP_KEY) references SALES_REPS(SALES_REP_KEY),
        CONTRACTS_TO_VENDORS as CONTRACTS(VENDOR_KEY) references VENDORS(VENDOR_KEY),
        CUSTOMERS_TO_REGIONS as CUSTOMERS(REGION_KEY) references REGIONS(REGION_KEY)
    )
    facts (
        CONTRACTS.CONTRACT_AMOUNT as AMOUNT comment='Contract value in EUR',
        CONTRACTS.CONTRACT_UNITS as UNITS comment='kWh for tariffs or unit count for hardware',
        CONTRACTS.CONTRACT_RECORD as 1 comment='Count of contracts'
    )
    dimensions (
        CUSTOMERS.CUSTOMER_KEY as CUSTOMER_KEY,
        CUSTOMERS.CUSTOMER_NAME as customer_name with synonyms=('Kunde','Name') comment='Customer name',
        CUSTOMERS.CUSTOMER_TYPE as customer_type with synonyms=('Kundentyp','Segment') comment='Privatkunde, Kleingewerbe, Gewerbekunde',
        CUSTOMERS.HOUSING_TYPE as housing_type with synonyms=('Wohnform','Gebaeudetyp') comment='Einfamilienhaus, Reihenhaus, Wohnung, etc.',
        CUSTOMERS.CITY as city with synonyms=('Stadt','Ort') comment='German city',
        CUSTOMERS.STATE as state with synonyms=('Region','Bundesland') comment='North, South, West, East',
        PRODUCTS.PRODUCT_KEY as PRODUCT_KEY,
        PRODUCTS.PRODUCT_NAME as product_name with synonyms=('Produkt','Tarif') comment='Product/tariff name',
        PRODUCTS.CATEGORY_NAME as category_name with synonyms=('Kategorie') comment='Product category',
        PRODUCTS.VERTICAL as vertical with synonyms=('Bereich','Segment') comment='Energy, Future Energy, Smart Home, E-Mobility',
        PRODUCT_CATEGORIES.CATEGORY_KEY as CATEGORY_KEY,
        PRODUCT_CATEGORIES.CATEGORY_NAME as MAIN_CATEGORY with synonyms=('Produktkategorie','product_category','main_category') comment='Electricity, Gas, Solar, Heat Pumps, Smart Home, E-Mobility',
        REGIONS.REGION_KEY as REGION_KEY,
        REGIONS.REGION_NAME as region_name with synonyms=('Region','Gebiet') comment='German region',
        CONTRACTS.SALE_ID as SALE_ID,
        CONTRACTS.DATE as CONTRACT_DATE with synonyms=('Vertragsdatum','Datum') comment='Contract date',
        CONTRACTS.SALES_REP_KEY as SALES_REP_KEY,
        SALES_REPS.REP_NAME as rep_name with synonyms=('Berater','Vertriebsmitarbeiter') comment='Energy consultant name',
        VENDORS.VENDOR_KEY as VENDOR_KEY,
        VENDORS.VENDOR_NAME as vendor_name with synonyms=('Partner','Installateur') comment='Installation partner'
    )
    metrics (
        CONTRACTS.TOTAL_REVENUE as SUM(contracts.contract_amount) comment='Total contract revenue in EUR',
        CONTRACTS.TOTAL_CONTRACTS as COUNT(contracts.contract_record) comment='Total number of contracts',
        CONTRACTS.AVERAGE_CONTRACT_VALUE as AVG(contracts.contract_amount) comment='Average contract value',
        CONTRACTS.TOTAL_UNITS as SUM(contracts.contract_units) comment='Total units sold'
    )
    comment='Semantic view for energy sales analysis - contracts, products, customers';

In [ ]:
-- BILLING AND CONSUMPTION SEMANTIC VIEW
CREATE OR REPLACE SEMANTIC VIEW ENERGY_AI_DEMO.ENERGY_SCHEMA.BILLING_SEMANTIC_VIEW
    tables (
        CUSTOMERS as ENERGY_AI_DEMO.ENERGY_SCHEMA.CUSTOMER_DIM primary key (CUSTOMER_KEY)
            with synonyms=('Kunden','customers')
            comment='Energy customers',
        BILLING as ENERGY_AI_DEMO.ENERGY_SCHEMA.BILLING_HISTORY primary key (BILLING_ID)
            with synonyms=('Rechnungen','Abrechnungen','billing','invoices')
            comment='Monthly energy billing and consumption'
    )
    relationships (
        BILLING_TO_CUSTOMERS as BILLING(CUSTOMER_KEY) references CUSTOMERS(CUSTOMER_KEY)
    )
    facts (
        BILLING.CONSUMPTION as CONSUMPTION_KWH comment='Energy consumption in kWh',
        BILLING.BILLING_AMOUNT as AMOUNT comment='Invoice amount in EUR',
        BILLING.BILLING_RECORD as 1 comment='Count of billing records'
    )
    dimensions (
        CUSTOMERS.CUSTOMER_KEY as CUSTOMER_KEY,
        CUSTOMERS.CUSTOMER_NAME as customer_name comment='Customer name',
        CUSTOMERS.HOUSING_TYPE as housing_type comment='Housing type',
        CUSTOMERS.CITY as city comment='City',
        BILLING.BILLING_ID as BILLING_ID,
        BILLING.BILLING_DATE as billing_date with synonyms=('Rechnungsdatum','Abrechnungsdatum') comment='Billing date',
        BILLING.BILLING_TYPE as billing_type with synonyms=('Energieart','Art') comment='Electricity or Gas',
        BILLING.PAYMENT_STATUS as payment_status with synonyms=('Zahlungsstatus','Status') comment='Bezahlt, Offen, Ueberfaellig'
    )
    metrics (
        BILLING.TOTAL_CONSUMPTION as SUM(billing.consumption) comment='Total consumption in kWh',
        BILLING.AVERAGE_CONSUMPTION as AVG(billing.consumption) comment='Average consumption in kWh',
        BILLING.TOTAL_BILLING_AMOUNT as SUM(billing.billing_amount) comment='Total billing amount',
        BILLING.TOTAL_INVOICES as COUNT(billing.billing_record) comment='Number of invoices'
    )
    comment='Semantic view for energy consumption and billing analysis';

In [ ]:
-- SERVICE LOGS SEMANTIC VIEW
CREATE OR REPLACE SEMANTIC VIEW ENERGY_AI_DEMO.ENERGY_SCHEMA.SERVICE_SEMANTIC_VIEW
    tables (
        CUSTOMERS as ENERGY_AI_DEMO.ENERGY_SCHEMA.CUSTOMER_DIM primary key (CUSTOMER_KEY)
            with synonyms=('Kunden','customers')
            comment='Energy customers',
        SERVICE_TICKETS as ENERGY_AI_DEMO.ENERGY_SCHEMA.SERVICE_LOGS primary key (LOG_ID)
            with synonyms=('Tickets','Anfragen','service requests','Kundenservice')
            comment='Customer service tickets and support requests'
    )
    relationships (
        SERVICE_TO_CUSTOMERS as SERVICE_TICKETS(CUSTOMER_KEY) references CUSTOMERS(CUSTOMER_KEY)
    )
    facts (
        SERVICE_TICKETS.TICKET_RECORD as 1 comment='Count of service tickets'
    )
    dimensions (
        CUSTOMERS.CUSTOMER_KEY as CUSTOMER_KEY,
        CUSTOMERS.CUSTOMER_NAME as customer_name comment='Customer name',
        CUSTOMERS.CITY as city comment='City',
        SERVICE_TICKETS.LOG_ID as TICKET_ID,
        SERVICE_TICKETS.LOG_DATE as ticket_date with synonyms=('Datum','Erstelldatum') comment='Ticket creation date',
        SERVICE_TICKETS.TOPIC as topic with synonyms=('Thema','Betreff') comment='Topic: Smart Meter, Rechnung, Waermepumpe, Solar, Tarif, Wallbox, Allgemein',
        SERVICE_TICKETS.CATEGORY as category with synonyms=('Kategorie') comment='Category: Installation, Abrechnung, Technisch, Vertrag, E-Mobility, Service',
        SERVICE_TICKETS.DESCRIPTION as description with synonyms=('Beschreibung','Details') comment='Ticket description',
        SERVICE_TICKETS.SENTIMENT as sentiment with synonyms=('Stimmung','Bewertung') comment='Customer sentiment: Positiv, Neutral, Negativ',
        SERVICE_TICKETS.CHANNEL as channel with synonyms=('Kanal','Kontaktweg') comment='Contact channel: Telefon, Email, Chat, App',
        SERVICE_TICKETS.PRIORITY as priority with synonyms=('Prioritaet','Dringlichkeit') comment='Priority: Niedrig, Mittel, Hoch, Kritisch',
        SERVICE_TICKETS.RESOLUTION_DATE as resolution_date with synonyms=('Loesungsdatum') comment='Resolution date'
    )
    metrics (
        SERVICE_TICKETS.TOTAL_TICKETS as COUNT(service_tickets.ticket_record) comment='Total number of tickets',
        SERVICE_TICKETS.NEGATIVE_TICKETS as SUM(CASE WHEN service_tickets.sentiment = 'Negativ' THEN 1 ELSE 0 END) comment='Negative sentiment tickets'
    )
    comment='Semantic view for customer service analysis';

In [ ]:
-- HR SEMANTIC VIEW
CREATE OR REPLACE SEMANTIC VIEW ENERGY_AI_DEMO.ENERGY_SCHEMA.HR_SEMANTIC_VIEW
    tables (
        DEPARTMENTS as ENERGY_AI_DEMO.ENERGY_SCHEMA.DEPARTMENT_DIM primary key (DEPARTMENT_KEY)
            with synonyms=('Abteilungen','departments')
            comment='Company departments',
        EMPLOYEES as ENERGY_AI_DEMO.ENERGY_SCHEMA.EMPLOYEE_DIM primary key (EMPLOYEE_KEY)
            with synonyms=('Mitarbeiter','employees','Personal')
            comment='Employees',
        HR_RECORDS as ENERGY_AI_DEMO.ENERGY_SCHEMA.HR_EMPLOYEE_FACT primary key (HR_FACT_ID)
            with synonyms=('HR-Daten','hr records')
            comment='HR fact records',
        JOBS as ENERGY_AI_DEMO.ENERGY_SCHEMA.JOB_DIM primary key (JOB_KEY)
            with synonyms=('Stellen','jobs','Positionen')
            comment='Job positions',
        LOCATIONS as ENERGY_AI_DEMO.ENERGY_SCHEMA.LOCATION_DIM primary key (LOCATION_KEY)
            with synonyms=('Standorte','locations')
            comment='Office locations'
    )
    relationships (
        HR_TO_DEPARTMENTS as HR_RECORDS(DEPARTMENT_KEY) references DEPARTMENTS(DEPARTMENT_KEY),
        HR_TO_EMPLOYEES as HR_RECORDS(EMPLOYEE_KEY) references EMPLOYEES(EMPLOYEE_KEY),
        HR_TO_JOBS as HR_RECORDS(JOB_KEY) references JOBS(JOB_KEY),
        HR_TO_LOCATIONS as HR_RECORDS(LOCATION_KEY) references LOCATIONS(LOCATION_KEY)
    )
    facts (
        HR_RECORDS.ATTRITION_FLAG as ATTRITION_FLAG comment='1 = employee left, 0 = active',
        HR_RECORDS.EMPLOYEE_SALARY as SALARY comment='Salary in EUR',
        HR_RECORDS.HR_RECORD as 1 comment='HR record count',
        EMPLOYEES.EMPLOYEE_COUNT as 1 comment='Employee count'
    )
    dimensions (
        DEPARTMENTS.DEPARTMENT_KEY as DEPARTMENT_KEY,
        DEPARTMENTS.DEPARTMENT_NAME as department_name with synonyms=('Abteilung') comment='Department name',
        EMPLOYEES.EMPLOYEE_KEY as EMPLOYEE_KEY,
        EMPLOYEES.EMPLOYEE_NAME as employee_name with synonyms=('Mitarbeiter','Name') comment='Employee name',
        EMPLOYEES.GENDER as gender with synonyms=('Geschlecht') comment='M or F',
        EMPLOYEES.HIRE_DATE as hire_date with synonyms=('Eintrittsdatum') comment='Hire date',
        HR_RECORDS.DATE as record_date comment='HR record date',
        JOBS.JOB_KEY as JOB_KEY,
        JOBS.JOB_TITLE as job_title with synonyms=('Position','Stelle') comment='Job title',
        JOBS.JOB_LEVEL as job_level with synonyms=('Ebene','Level') comment='Job level',
        LOCATIONS.LOCATION_KEY as LOCATION_KEY,
        LOCATIONS.LOCATION_NAME as location_name with synonyms=('Standort') comment='Office location'
    )
    metrics (
        HR_RECORDS.TOTAL_SALARY as SUM(hr_records.employee_salary) comment='Total salary cost',
        HR_RECORDS.AVG_SALARY as AVG(hr_records.employee_salary) comment='Average salary',
        HR_RECORDS.ATTRITION_COUNT as SUM(hr_records.attrition_flag) comment='Attrition count',
        EMPLOYEES.TOTAL_EMPLOYEES as COUNT(employees.employee_count) comment='Total employees'
    )
    comment='Semantic view for HR analytics';

In [ ]:
-- Verify semantic views\nSHOW SEMANTIC VIEWS;

---

## 9. Unstructured Data Processing <a id="9-unstructured-data"></a>

### Concept: Cortex Parse Document

**Cortex Parse** is a Snowflake feature that extracts text and structure from unstructured documents:

- **PDF documents**: Contracts, guides, policies
- **Markdown files**: Documentation, FAQs
- **Layout preservation**: Maintains document structure

**Modes:**
- `OCR`: Optical character recognition for scanned documents
- `LAYOUT`: Preserves document layout and structure (recommended)

This parsed content becomes searchable via **Cortex Search**.

In [ ]:
-- Parse PDF and Markdown documents
CREATE OR REPLACE TABLE parsed_content AS 
SELECT 
    relative_path,
    BUILD_STAGE_FILE_URL('@ENERGY_AI_DEMO.ENERGY_SCHEMA.ENERGY_STAGE', relative_path) as file_url,
    TO_FILE(BUILD_STAGE_FILE_URL('@ENERGY_AI_DEMO.ENERGY_SCHEMA.ENERGY_STAGE', relative_path)) file_object,
    SNOWFLAKE.CORTEX.PARSE_DOCUMENT(
        @ENERGY_AI_DEMO.ENERGY_SCHEMA.ENERGY_STAGE,
        relative_path,
        {'mode':'LAYOUT'}
    ):content::string as content
FROM directory(@ENERGY_AI_DEMO.ENERGY_SCHEMA.ENERGY_STAGE) 
WHERE relative_path ILIKE 'unstructured_docs/%.pdf' 
   OR relative_path ILIKE 'unstructured_docs/%.md';

In [ ]:
-- Verify parsed documents
SELECT relative_path, LEFT(content, 200) as content_preview 
FROM parsed_content 
LIMIT 5;

---

## 10. Cortex Search Services <a id="10-cortex-search"></a>

### Concept: Cortex Search

**Cortex Search** provides vector-based semantic search over your data:

- Uses embedding models to convert text to vectors
- Finds semantically similar content (not just keyword matching)
- Ideal for RAG (Retrieval Augmented Generation) applications

**Key Parameters:**
- `ON`: The column to search (the content)
- `ATTRIBUTES`: Metadata columns to return
- `TARGET_LAG`: How often to refresh the index
- `EMBEDDING_MODEL`: The model for vectorization

We create **4 search services** for different document types.

In [ ]:
-- Search service for energy policy documents
CREATE OR REPLACE CORTEX SEARCH SERVICE Search_energy_docs
    ON content
    ATTRIBUTES relative_path, file_url, title
    WAREHOUSE = ENERGY_INTELLIGENCE_DEMO_WH
    TARGET_LAG = '30 day'
    EMBEDDING_MODEL = 'snowflake-arctic-embed-l-v2.0'
    AS (
        SELECT
            relative_path,
            file_url,
            REGEXP_SUBSTR(relative_path, '[^/]+$') as title,
            content
        FROM parsed_content
        WHERE relative_path ILIKE '%/energy/%'
    );

In [ ]:
-- Search service for product documents
CREATE OR REPLACE CORTEX SEARCH SERVICE Search_product_docs
    ON content
    ATTRIBUTES relative_path, file_url, title
    WAREHOUSE = ENERGY_INTELLIGENCE_DEMO_WH
    TARGET_LAG = '30 day'
    EMBEDDING_MODEL = 'snowflake-arctic-embed-l-v2.0'
    AS (
        SELECT
            relative_path,
            file_url,
            REGEXP_SUBSTR(relative_path, '[^/]+$') as title,
            content
        FROM parsed_content
        WHERE relative_path ILIKE '%/products/%'
    );

In [ ]:
-- Search service for customer service documents
CREATE OR REPLACE CORTEX SEARCH SERVICE Search_service_docs
    ON content
    ATTRIBUTES relative_path, file_url, title
    WAREHOUSE = ENERGY_INTELLIGENCE_DEMO_WH
    TARGET_LAG = '30 day'
    EMBEDDING_MODEL = 'snowflake-arctic-embed-l-v2.0'
    AS (
        SELECT
            relative_path,
            file_url,
            REGEXP_SUBSTR(relative_path, '[^/]+$') as title,
            content
        FROM parsed_content
        WHERE relative_path ILIKE '%/service/%'
    );

In [ ]:
-- Search service for service logs (structured data search)
CREATE OR REPLACE CORTEX SEARCH SERVICE Search_service_logs
    ON description
    ATTRIBUTES log_id, customer_key, topic, category, sentiment, priority
    WAREHOUSE = ENERGY_INTELLIGENCE_DEMO_WH
    TARGET_LAG = '1 day'
    EMBEDDING_MODEL = 'snowflake-arctic-embed-l-v2.0'
    AS (
        SELECT
            log_id,
            customer_key,
            topic,
            category,
            description,
            sentiment,
            priority
        FROM service_logs
    );

In [ ]:
-- Verify search services\nSHOW CORTEX SEARCH SERVICES;

---

## 11. Snowflake Intelligence Agent <a id="11-agent"></a>

### Concept: Snowflake Intelligence Agent

A **Snowflake Intelligence Agent** is an AI orchestrator that can:

1. **Understand natural language** - Process questions in German or English
2. **Select appropriate tools** - Choose between Text-to-SQL, Search, Web Scraping
3. **Execute multi-step tasks** - Chain tool calls together
4. **Return formatted responses** - With data, visualizations, and citations

### Setting up External Access (for Web Scraping)

In [ ]:
-- Create network rule for web access
CREATE OR REPLACE NETWORK RULE Energy_Intelligence_WebAccessRule
    MODE = EGRESS
    TYPE = HOST_PORT
    VALUE_LIST = ('0.0.0.0:80', '0.0.0.0:443');

In [ ]:
-- Create external access integration (requires ACCOUNTADMIN)
USE ROLE accountadmin;

GRANT ALL PRIVILEGES ON DATABASE ENERGY_AI_DEMO TO ROLE ACCOUNTADMIN;
GRANT ALL PRIVILEGES ON SCHEMA ENERGY_AI_DEMO.ENERGY_SCHEMA TO ROLE ACCOUNTADMIN;
GRANT USAGE ON NETWORK RULE ENERGY_AI_DEMO.ENERGY_SCHEMA.Energy_Intelligence_WebAccessRule TO ROLE accountadmin;

USE SCHEMA ENERGY_AI_DEMO.ENERGY_SCHEMA;

CREATE OR REPLACE EXTERNAL ACCESS INTEGRATION Energy_Intelligence_ExternalAccess_Integration
    ALLOWED_NETWORK_RULES = (Energy_Intelligence_WebAccessRule)
    ENABLED = true;

GRANT USAGE ON INTEGRATION Energy_Intelligence_ExternalAccess_Integration TO ROLE Energy_Intelligence_Demo;

In [ ]:
-- Create helper functions
USE ROLE Energy_Intelligence_Demo;

-- Presigned URL procedure for file access
CREATE OR REPLACE PROCEDURE Get_File_Presigned_URL_SP(
    RELATIVE_FILE_PATH STRING, 
    EXPIRATION_MINS INTEGER DEFAULT 60
)
RETURNS STRING
LANGUAGE SQL
COMMENT = 'Generates a presigned URL for files in ENERGY_STAGE'
EXECUTE AS CALLER
AS
$$
DECLARE
    presigned_url STRING;
    sql_stmt STRING;
    expiration_seconds INTEGER;
    stage_name STRING DEFAULT '@ENERGY_AI_DEMO.ENERGY_SCHEMA.ENERGY_STAGE';
BEGIN
    expiration_seconds := EXPIRATION_MINS * 60;
    sql_stmt := 'SELECT GET_PRESIGNED_URL(' || stage_name || ', ' || '''' || RELATIVE_FILE_PATH || '''' || ', ' || expiration_seconds || ') AS url';
    EXECUTE IMMEDIATE :sql_stmt;
    SELECT "URL" INTO :presigned_url FROM TABLE(RESULT_SCAN(LAST_QUERY_ID()));
    RETURN :presigned_url;
END;
$$;

In [ ]:
-- Web scraping function
CREATE OR REPLACE FUNCTION Web_scrape(weburl STRING)
RETURNS STRING
LANGUAGE PYTHON
RUNTIME_VERSION = 3.11
HANDLER = 'get_page'
EXTERNAL_ACCESS_INTEGRATIONS = (Energy_Intelligence_ExternalAccess_Integration)
PACKAGES = ('requests', 'beautifulsoup4')
AS
$$
import requests
from bs4 import BeautifulSoup

def get_page(weburl):
    response = requests.get(weburl)
    soup = BeautifulSoup(response.text)
    return soup.get_text()
$$;

### Creating the Intelligence Agent

The agent specification defines:
- **Instructions**: How the agent should behave and respond
- **Tools**: Available capabilities (Cortex Analyst, Cortex Search, Functions)
- **Tool Resources**: Configuration for each tool

In [ ]:
-- Create the Snowflake Intelligence Agent
CREATE OR REPLACE AGENT ENERGY_AI_DEMO.ENERGY_SCHEMA.Energy_Chatbot_Agent
WITH PROFILE='{ "display_name": "EPOWER Energy Intelligence Agent" }'
COMMENT=$$ Agent for energy retail analysis: contracts, consumption, service tickets, and documents. $$
FROM SPECIFICATION $$
{
  "models": {
    "orchestration": ""
  },
  "instructions": {
    "response": "Du bist ein Datenanalyst fuer EPOWER Energie Deutschland. Du hast Zugriff auf Energie-Vertriebsdaten (Strom, Gas, Solar, Waermepumpen, Smart Home, E-Mobility), Verbrauchsabrechnungen, Kundenservice-Tickets und interne Dokumente. Antworte auf Deutsch, wenn der Nutzer auf Deutsch fragt. Liefere Visualisierungen wenn moeglich.",
    "orchestration": "Nutze Cortex Search fuer Dokumente und Cortex Analyst fuer strukturierte Datenanalysen. Bei Fragen zu Verbrauch, nutze das Billing Datamart. Bei Fragen zu Service-Tickets oder Beschwerden, nutze das Service Datamart. Bei Produktfragen, nutze das Energy Sales Datamart.",
    "sample_questions": [
      {"question": "Was ist der durchschnittliche Stromverbrauch fuer Kunden mit Waermepumpen in Hamburg?"},
      {"question": "Zeige mir alle negativen Service-Tickets zum Thema Smart Meter."},
      {"question": "Welche Produkte wurden letzten Monat am meisten verkauft?"}
    ]
  },
  "tools": [
    {"tool_spec": {"type": "cortex_analyst_text_to_sql", "name": "Query Energy Sales", "description": "Analysiert Energievertraege, Produkte (Strom, Gas, Solar, Waermepumpen), Kunden und Regionen."}},
    {"tool_spec": {"type": "cortex_analyst_text_to_sql", "name": "Query Billing Data", "description": "Analysiert Energieverbrauch (kWh), Abrechnungen und Zahlungsstatus fuer Strom und Gas."}},
    {"tool_spec": {"type": "cortex_analyst_text_to_sql", "name": "Query Service Tickets", "description": "Analysiert Kundenservice-Tickets, Beschwerden, Sentiment und Themen wie Smart Meter, Waermepumpe, Solar."}},
    {"tool_spec": {"type": "cortex_analyst_text_to_sql", "name": "Query HR Data", "description": "Analysiert HR-Daten: Mitarbeiter, Gehaelter, Fluktuation, Abteilungen."}},
    {"tool_spec": {"type": "cortex_search", "name": "Search Energy Documents", "description": "Sucht in Energiedokumenten: AGBs, Foerderungen, Richtlinien."}},
    {"tool_spec": {"type": "cortex_search", "name": "Search Product Documents", "description": "Sucht in Produktdokumenten: Waermepumpen-Guide, Solar-Anleitungen, Smart Meter, E-Mobility."}},
    {"tool_spec": {"type": "cortex_search", "name": "Search Service Documents", "description": "Sucht in Service-Dokumenten: Rechnungserklaerungen, Energiespar-Tipps, Kundenservice-Handbuch."}},
    {"tool_spec": {"type": "cortex_search", "name": "Search Service Logs", "description": "Semantische Suche in Service-Tickets nach Beschreibungen und Themen."}},
    {"tool_spec": {"type": "generic", "name": "Web_scraper", "description": "Analysiert Inhalte von Webseiten (z.B. fuer aktuelle Energiepreise, Foerderprogramme).", "input_schema": {"type": "object", "properties": {"weburl": {"description": "URL der Webseite", "type": "string"}}, "required": ["weburl"]}}},
    {"tool_spec": {"type": "generic", "name": "Dynamic_Doc_URL_Tool", "description": "Generiert Download-URLs fuer Dokumente aus dem Stage.", "input_schema": {"type": "object", "properties": {"expiration_mins": {"description": "Gueltigkeit in Minuten (Standard: 5)", "type": "number"}, "relative_file_path": {"description": "Relativer Pfad zur Datei", "type": "string"}}, "required": ["relative_file_path"]}}}
  ],
  "tool_resources": {
    "Query Energy Sales": {"semantic_view": "ENERGY_AI_DEMO.ENERGY_SCHEMA.ENERGY_SALES_SEMANTIC_VIEW"},
    "Query Billing Data": {"semantic_view": "ENERGY_AI_DEMO.ENERGY_SCHEMA.BILLING_SEMANTIC_VIEW"},
    "Query Service Tickets": {"semantic_view": "ENERGY_AI_DEMO.ENERGY_SCHEMA.SERVICE_SEMANTIC_VIEW"},
    "Query HR Data": {"semantic_view": "ENERGY_AI_DEMO.ENERGY_SCHEMA.HR_SEMANTIC_VIEW"},
    "Search Energy Documents": {"id_column": "RELATIVE_PATH", "max_results": 5, "name": "ENERGY_AI_DEMO.ENERGY_SCHEMA.SEARCH_ENERGY_DOCS", "title_column": "TITLE"},
    "Search Product Documents": {"id_column": "RELATIVE_PATH", "max_results": 5, "name": "ENERGY_AI_DEMO.ENERGY_SCHEMA.SEARCH_PRODUCT_DOCS", "title_column": "TITLE"},
    "Search Service Documents": {"id_column": "RELATIVE_PATH", "max_results": 5, "name": "ENERGY_AI_DEMO.ENERGY_SCHEMA.SEARCH_SERVICE_DOCS", "title_column": "TITLE"},
    "Search Service Logs": {"id_column": "LOG_ID", "max_results": 10, "name": "ENERGY_AI_DEMO.ENERGY_SCHEMA.SEARCH_SERVICE_LOGS", "title_column": "TOPIC"},
    "Web_scraper": {"execution_environment": {"type": "warehouse", "warehouse": "ENERGY_INTELLIGENCE_DEMO_WH"}, "identifier": "ENERGY_AI_DEMO.ENERGY_SCHEMA.WEB_SCRAPE", "name": "WEB_SCRAPE(VARCHAR)", "type": "function"},
    "Dynamic_Doc_URL_Tool": {"execution_environment": {"type": "warehouse", "warehouse": "ENERGY_INTELLIGENCE_DEMO_WH"}, "identifier": "ENERGY_AI_DEMO.ENERGY_SCHEMA.GET_FILE_PRESIGNED_URL_SP", "name": "GET_FILE_PRESIGNED_URL_SP(VARCHAR, DEFAULT NUMBER)", "type": "procedure"}
  }
}
$$;

In [ ]:
-- Assign agent to Snowflake Intelligence and grant access
USE ROLE accountadmin;

ALTER SNOWFLAKE INTELLIGENCE snowflake_intelligence_object_default ADD AGENT ENERGY_AI_DEMO.ENERGY_SCHEMA.Energy_Chatbot_Agent;
GRANT USAGE ON AGENT ENERGY_AI_DEMO.ENERGY_SCHEMA.Energy_Chatbot_Agent TO ROLE PUBLIC;
GRANT USAGE ON AGENT ENERGY_AI_DEMO.ENERGY_SCHEMA.Energy_Chatbot_Agent TO ROLE Energy_Intelligence_Demo;

USE ROLE Energy_Intelligence_Demo;

---

## 12. Verification & Next Steps <a id="12-verification"></a>

### Setup Complete!

Let's verify everything was created successfully.

In [ ]:
-- Final verification
SELECT 'Energy AI Demo Setup Complete!' as status;
SELECT 'Database: ENERGY_AI_DEMO.ENERGY_SCHEMA' as info;
SELECT 'Agent: Energy_Chatbot_Agent' as info;
SELECT 'Semantic Views: 4 (Energy Sales, Billing, Service, HR)' as info;
SELECT 'Cortex Search: 4 services (Energy, Product, Service docs + Service logs)' as info;

In [ ]:
-- View all created objects
SHOW TABLES IN SCHEMA ENERGY_AI_DEMO.ENERGY_SCHEMA;
SHOW SEMANTIC VIEWS IN SCHEMA ENERGY_AI_DEMO.ENERGY_SCHEMA;
SHOW CORTEX SEARCH SERVICES IN SCHEMA ENERGY_AI_DEMO.ENERGY_SCHEMA;
SHOW AGENTS IN SCHEMA ENERGY_AI_DEMO.ENERGY_SCHEMA;

### Try the Agent!

Now you can interact with the Energy Intelligence Agent. Here are some sample questions to try:

**Sales & Products (German):**
- "Welche Produkte wurden letzten Monat am meisten verkauft?"
- "Zeige mir den Umsatz nach Region fuer 2024"

**Consumption & Billing:**
- "What is the average electricity consumption for customers with heat pumps?"
- "Wie ist der Zahlungsstatus unserer Rechnungen aufgeteilt?"

**Service Tickets:**
- "Zeige mir alle negativen Service-Tickets zum Thema Smart Meter"
- "What are the most common service ticket topics?"

**Document Search (RAG):**
- "Was sind die Voraussetzungen fuer die Waermepumpen-Foerderung?"
- "Erklaere mir, wie ich meine Stromrechnung lesen kann"

### What You've Learned

| Concept | What You Did |
|---------|-------------|
| **Git Integration** | Connected to GitHub, loaded data directly from Git stage |
| **Star Schema** | Created dimension and fact tables for analytical workloads |
| **Semantic Views** | Defined business vocabulary, relationships, and metrics for AI |
| **Cortex Parse** | Extracted text from PDF documents |
| **Cortex Search** | Created vector search indexes for RAG |
| **Intelligence Agent** | Built a multi-tool AI assistant |

### Clean Up (Optional)

To remove all demo resources:

```sql
USE ROLE accountadmin;
DROP DATABASE IF EXISTS ENERGY_AI_DEMO;
DROP WAREHOUSE IF EXISTS ENERGY_INTELLIGENCE_DEMO_WH;
DROP ROLE IF EXISTS Energy_Intelligence_Demo;
DROP API INTEGRATION IF EXISTS git_api_integration_energy;
DROP EXTERNAL ACCESS INTEGRATION IF EXISTS Energy_Intelligence_ExternalAccess_Integration;
```